At this point the content has already been split into paragraphs, divided into Russian and English with language detection, as well as the Russian content is translated to English.

In [ ]:
#!pip3 install itables
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)
#maxBytes=0

In [ ]:
#!pip3 install --upgrade numpy
#!pip3 install numba==0.53
#!pip3 install --upgrade guesslang

In [ ]:
#!pip3 install --upgrade pandas
#!pip3 install pickle5
#!pip3 install --upgrade nltk
#!pip3 install --upgrade textblob
#!pip3 install ipython-autotime
#!pip3 install umap-learn[plot]
#!pip3 install hdbscan
#!pip3 uninstall hdbscan --yes
#!pip3 install hdbscan --no-cache-dir --no-binary :all:
import nltk
#nltk.download('punkt')
#nltk.download('wordnet')
#nltk.download('omw-1.4')
#nltk.download('brown')
#nltk.download('averaged_perceptron_tagger')
#nltk.download('vader_lexicon')
#!pip3 install --upgrade requests
#!pip3 install sentence_transformers
#!pip3 install tensorflow
#!pip3 install tensorflow_hub
#!pip3 install --upgrade tensorflow #==2.9.0
#!pip3 install spaCy
#!python3 -m spacy download en_core_web_sm
#!pip3 install pandarallel
#!pip3 install imagehash
#!pip3 install simplejson

#plots
#!pip3 install matplotlib
#!pip3 install datashader
#!pip3 install bokeh
#!pip3 install holoviews
#!pip3 install pyenchant

#!python -m pip install hdbscan --no-cache-dir --no-binary :all:

In [ ]:
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn', disables warning windows
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false" # disables huggingface warnings

import numpy as np
import pickle5 as pickle
import re
from nltk.stem import WordNetLemmatizer
import nltk
from textblob import TextBlob
from tqdm.notebook import tqdm
tqdm.pandas()
import requests
from pandarallel import pandarallel
import umap.umap_ as umap
import hdbscan
import imagehash
import spacy
from collections import Counter
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import random
import itertools
import json
#from google.colab import drive
#import simplejson

# ML
import tensorflow as tf
import tensorflow_hub as hub
from sentence_transformers import SentenceTransformer

from scipy import spatial

# to plot
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import umap
%matplotlib inline
#import umap.plot
import random
import enchant

# Functions

### Text preparation & cleaning

In [ ]:
def load_dfs():
    if use_saved_dataframes == False:
        dataframe_all_forums = pd.DataFrame()

        # execute forumdataToDataframe, which gets the thread data from each csv file and transforms is to a dataframe
        # concatenate the new dataframe to one big dataframe
        for forum_id in forums_to_process:
            df_dict[forum_id] = forumdataToDataframe(forum_id)
            df2= pd.read_pickle(f'{path_inputs}userids/{forum_id}_df_userids_threadids.pkl')
            df_dict[forum_id] = pd.merge(df_dict[forum_id], df2, on = "thread_nr", how = "left")

        for forum_id in forums_to_process:
            dataframe_all_forums = pd.concat([dataframe_all_forums, df_dict[forum_id]],ignore_index=True, axis=0)

    return dataframe_all_forums, df_dict

In [ ]:
#gets the threaddata from each csv file and transforms is to a dataframe
def forumdataToDataframe(forum_nr): # if forumdata is a csv of sentence, thread_nr, timestamp
    dataframe_ru = pd.read_csv(f'{path_inputs}split_newlines_output/'+str(forum_nr)+'_translated_ru_threads.txt', lineterminator='\n', sep = '\t', header=None)
    dataframe_en = pd.read_csv(f'{path_inputs}split_newlines_output/'+str(forum_nr)+'_en_threads.txt', lineterminator='\n', sep = '\t', header=None)
    dataframe_ru.columns = ['body', 'thread_nr', 'timestamp']
    dataframe_ru['language'] = "russian"
    dataframe_en.columns = ['body', 'thread_nr', 'timestamp']
    dataframe_en['language'] = "english"
    dataframe_both_lang = pd.concat([dataframe_ru, dataframe_en],ignore_index=True, axis=0)
    dataframe_both_lang['forum_nr'] = forum_nr
    dataframe_both_lang['timestamp'] = pd.to_datetime(dataframe_both_lang['timestamp'])
    return dataframe_both_lang

In [ ]:
punctuations = """!"%()'*+,*/:#»«";“<=>?[\]^`”{.}~-_|$@"""
numbers = "0123456789"
def clean_text_local(row):
    # remove sentences that contain more that 90% numbers
        
    
    row=re.sub("-?NEWLINE_TOKEN", " ",row)
    row=re.sub("TAB_TOKEN", " ",row)
    row=re.sub("Alternate option=", "",row)
    row=re.sub('RT','',row)
    row = row.lower()

    row=re.sub('\x0c',' ',row)
    row=re.sub('\u200c',' ',row)
    row=re.sub('\\\\n',' ',row)
    row=re.sub('\n',' ',row)

    row=re.sub("@[A-Za-z0-9]+","",row)
    row=re.sub("http\S+|www.\S+","",row) # removes the urls
    row = re.sub(r'<.*?>', '', row)
    row=re.sub("&amp","&",row)

    no_punct = ""
    for char in row:
        if char not in punctuations:
            no_punct = no_punct + char
    return no_punct


def check_percent_nonalphabet(value, threshold):
    amount_of_non_alphabet = 0
    for char in value:
        if char.isalpha() == False:
            amount_of_non_alphabet = amount_of_non_alphabet + 1
    if (amount_of_non_alphabet/len(value))*100 >= threshold:  #check if min %threshold are non-alpha char
        return True
    else:
        return False

def cleanRowsWithPercentageOfNumbers(dataframe, threshold):
    for index, row in dataframe.iterrows():
        if check_percent_nonalphabet(row['body'], threshold) == True:
            dataframe.drop([index], inplace = True)
    return dataframe


def check_punctuation_percent_nr(value, threshold):
    amount_of_punctuations = 0
    for char in value:
        if char in punctuations:
            amount_of_punctuations = amount_of_punctuations + 1
    if (amount_of_punctuations/len(value))*100 > threshold:  #check if max threshold% are punctuations
        return True
    else:
        return False

def cleanRowsWithPercentageOfPunctuations(dataframe, threshold):
    dropped_rows = []
    for index, row in dataframe.iterrows():
        if check_punctuation_percent_nr(row['body'], threshold) == True: # more than threshold are punctuations => sorted out
            dropped_rows.append(row["body"])
            dataframe.drop([index], inplace = True)
    return dataframe, dropped_rows

## ML Functions

In [ ]:
def embed(model, model_type, sentences):
  if model_type == 'use':
        embeddings = model(list(sentences))
  elif model_type == 'sentence transformer':
        embeddings = model.encode(list(sentences),show_progress_bar=True)
    
  return embeddings

In [ ]:
def splitEmbed(model_, model_type_, all_sentences_, n_partitions_=20):
    all_sentences_ = np.array_split(all_sentences_, indices_or_sections = n_partitions_)
    for i, new_sentences in enumerate(all_sentences_):
        if i==0:
            all_embeddings = embed(model_, model_type_, new_sentences)
        if i>0:
            new_embeddings = embed(model_, model_type_, new_sentences)
            all_embeddings = np.concatenate((all_embeddings,new_embeddings),axis=0)
    return all_embeddings

In [ ]:
def vectorToColumn(df,vectors,column_name):
    df[column_name]=0
    df[column_name]=df[column_name].astype(object)
    df.reset_index(drop=True,inplace=True)
    indices=df.index
    for i in indices:
        df.at[i, column_name]=vectors[i]
    return df

In [ ]:
nlp=spacy.load('en_core_web_sm')

def extract_labels(category_docs):
    """
    Extract labels from documents in the same cluster by concatenating
    most common verbs, ojects, and nouns
    """

    verbs = []
    dobjs = []
    nouns = []
    adjs = []
    
    verb = ''
    dobj = ''
    noun1 = ''
    noun2 = ''

    # for each document, append verbs, dobs, nouns, and adjectives to 
    # running lists for whole cluster
    for i in range(len(category_docs)):
        doc = nlp(category_docs[i])
        for token in doc:
            if token.is_stop==False:
                if token.dep_ == 'ROOT':
                    verbs.append(token.text.lower())

                elif token.dep_=='dobj':
                    dobjs.append(token.text.lower())

                elif token.pos_=='NOUN':
                    nouns.append(token.text.lower())
                    
                elif token.pos_=='ADJ':
                    adjs.append(token.text.lower())

    verbs = Counter(verbs)
    dobjs = Counter(dobjs)
    nouns = Counter(nouns)

    if len(verbs) > 0:
        verb = verbs.most_common(1)[0][0]

    if len(dobjs) > 0:
        dobj = dobjs.most_common(1)[0][0]

    if len(nouns) > 0:
        noun1 = nouns.most_common(1)[0][0]

    if len(set(nouns)) > 1:
        noun2 = nouns.most_common(2)[1][0]
    
    # concatenate the most common verb-dobj-noun1-noun2 (if they exist)
    label_words = [verb, dobj]
    
    for word in [noun1, noun2]:
        if word not in label_words:
            label_words.append(word)
    
    if '' in label_words:
        label_words.remove('')
    
    label = '_'.join(label_words)
    
    return label

In [ ]:
def generateUmapAndCluster(message_embeddings,
                      n_neighbors,
                      n_components,
                      min_cluster_size,
                      random_state = None):
    """
    Generate HDBSCAN cluster object after reducing embedding dimensionality with UMAP
    """
    # reduces dimensionality
    reducer = (umap.UMAP(#n_neighbors=n_neighbors, 
                                n_components = n_components, 
                                metric='cosine', 
                                random_state=random_state))

    umap_embeddings = reducer.fit_transform(message_embeddings)
    print(umap_embeddings.shape)
    
    
    '''Density-based algorithms are a good option here as they do not require 
    specifying the number of clusters and are indifferent to cluster shape. 
    Hierarchical Density-Based Spatial Clustering of Applications with Noise (HDBSCAN)'''
    clusters = hdbscan.HDBSCAN(min_cluster_size = min_cluster_size,
                        metric='euclidean', 
                        cluster_selection_method='eom').fit_predict(umap_embeddings)


    return clusters, umap_embeddings


### Clustering Functions

In [ ]:
def text_cluster_to_label(data_):
  clusters_dict = dict()
  clusters_set = set(data_['cluster'])
  for cluster_i in clusters_set:
    label_ = extract_labels(list(data_[data_['cluster']==cluster_i]['body']))
    print(label_)
    clusters_dict.update({cluster_i:label_})
  return data_['cluster'].map(clusters_dict),clusters_dict


### Plotting functions

In [ ]:
def plot_test_best_n_neighbors():
    for n in (2, 5):#, 10, 20, 50, 100, 200):

      #for i, txt in enumerate(list(data['body'])):
      #    plt.annotate(txt, (x[i], y[i]))


      RU = embeddings_st1_ru
      manifold_ru = umap.UMAP(n_components=2,
                          n_neighbors = n,    
                          metric='cosine', verbose=True, 
                          random_state=1).fit(RU)
      RU_reduced = manifold_ru.transform(RU)
      ru_x = [ru_x for ru_x, ru_y in RU_reduced]
      ru_y = [ru_y for ru_x, ru_y in RU_reduced]

      EN = embeddings_st1_en
      manifold_en = umap.UMAP(n_components=2, 
                          n_neighbors = n,
                          metric='cosine', verbose=True, 
                          random_state=1).fit(EN)
      EN_reduced = manifold_en.transform(EN)
      en_x = [en_x for en_x, en_y in EN_reduced]
      en_y = [en_y for en_x, en_y in EN_reduced]

      #plt.xlim(-5,18)
      #plt.ylim(0, 18)
      plt.scatter(ru_x, ru_y, s=0.1, c = "blue")
      plt.scatter(en_x, en_y, s=0.1, c = "red")

      plt.title('clusters title= n_neighbors = ' + str(n))
      plt.figure(n) # Here's the part I need

In [ ]:
# generates a 2 dimensional UMAP for plotting the data
def generateUmapForPlotting(embeddings, neighbors = 10):
    print("umapforplotting0")
    manifold = umap.UMAP(n_components=2, n_neighbors = neighbors, metric='cosine', verbose=True, random_state=1).fit(embeddings)
    print("umapforplotting1")
    embeddings_reduced = manifold.transform(embeddings)
    print("umapforplotting2")
    x = [x for x, y in embeddings_reduced]
    y = [y for x, y in embeddings_reduced]
    return x,y


# Function that plots a dataframe, either embeddings or x&y must be given
def plot_by_language( forum_nr, dataframe, embeddings = [None], xlim = None , ylim = None, thickness = 0.7):
    x,y = generateUmapForPlotting(embeddings)
    #PLOTTING
    sns.set_theme(font_scale=1.5, style = "white")
    plt.figure(figsize = (15,15)) # Here's the part I need
    if (xlim != None):
        xlim_min, xlim_max = xlim
        plt.xlim(xlim_min, xlim_max)
    if (ylim != None):
        ylim_min, ylim_max = ylim
        plt.ylim(ylim_min, ylim_max)
    plt.scatter(x, y, s=thickness, c = dataframe["language"].map(graph_colors_dict))
    plt.title(f'Clustering by language of Forum {forum_nr}')
    pop_en = mpatches.Patch(color=graph_colors_dict.get("english"), label="English")
    pop_ru = mpatches.Patch(color=graph_colors_dict.get("russian"), label='Russian')
    plt.legend(handles=[pop_en,pop_ru], loc="upper right")
    plt.savefig(f"{path_figures}forum_" + str(forum_nr) +"_color_by_language.png")
    

# START PROGRAM

In [ ]:
use_saved_dataframes = False #when set to true, the data will be loaded from inputs folder

In [ ]:
path_inputs = "/path/inputs/"
path_outputs = "/path/outputs/"
path_figures = "/path/figures/"

forums_to_process = [4,21,58] #[3, 4, 21, 58, 61, 64, 65, 71, 72]
subforums_to_recalc = [4,21,58]#[3, 4, 21, 58, 61, 64, 65, 71, 72]
df_dict= {}

In [ ]:
graph_colors_dict = {
    "english":"midnightblue",
    "russian":"cadetblue",
    4 : "salmon",
    "4_hatch" : "O",
    21 : "cadetblue",
    "21_hatch" : "*",
    58 : "midnightblue",
    "58_hatch" : "/",
}

forum_names_dict = {
    4:"IM messengers & social networks",
    21:"Games, Music, Movies",
    58:"Web development"
}

## Load translations into panda dataframe

In [ ]:
dataframe_all_forums, df_dict = load_dfs()

## Clean text

In [ ]:
dataframe_all_forums, dropped_rows = cleanRowsWithPercentageOfPunctuations(dataframe_all_forums, threshold = 12)
dataframe_all_forums['body']=dataframe_all_forums['body'].map(clean_text_local) # clean text
dataframe_all_forums = dataframe_all_forums[dataframe_all_forums["body"] != ""] # delete empty rows
dataframe_all_forums = cleanRowsWithPercentageOfNumbers(dataframe_all_forums, threshold = 100)  #clean strings with threshold% numbers


for forum_id in subforums_to_recalc:
    df_dict[forum_id], dropped_rows_subforum = cleanRowsWithPercentageOfPunctuations(df_dict[forum_id], threshold = 12)
    df_dict[forum_id]['body'] = df_dict[forum_id]["body"].map(clean_text_local) # clean text
    df_dict[forum_id] = df_dict[forum_id][df_dict[forum_id]["body"] != ""] # delete empty rows
    df_dict[forum_id] = cleanRowsWithPercentageOfNumbers(df_dict[forum_id], threshold = 100)  #clean strings with threshold% numbers

## Generate model and dataframe

In [ ]:
model_st1 = SentenceTransformer('all-mpnet-base-v2') #sentence-BERT model

embeddings_st1 = splitEmbed(model_=model_st1, model_type_='sentence transformer', all_sentences_=dataframe_all_forums['body'], n_partitions_ = round(len(list(dataframe_all_forums["body"]))/8000)) # devide the set so that one partition has about 5000-10000 partitions
dataframe_all_forums=vectorToColumn(df=dataframe_all_forums,vectors=embeddings_st1,column_name='all_mpnet_base_v2')

embeddings_dict = {}
for forum_id in subforums_to_recalc:
    print(f"Processing forum {str(forum_id)}")
    n_partitions = round(len(list(df_dict[forum_id]["body"]))/8000)
    if n_partitions == 0: n_partitions = 1
    embeddings_dict[forum_id] = splitEmbed(model_= model_st1, model_type_='sentence transformer', all_sentences_=df_dict[forum_id]['body'], n_partitions_ = n_partitions) # devide the set so that one partition has about 5000-10000 partitions
    df_dict[forum_id] = vectorToColumn(df=df_dict[forum_id], vectors=embeddings_dict[forum_id] ,column_name='all_mpnet_base_v2')


## Plotting

In [ ]:
font_scale = 2
#PLOTTING by FORUMS
def plot_all_forums(dataframe, embeddings_st1):
    figurename = ""
    for forum in forums_to_process:
        figurename = figurename + "_" + str(forum)
    print(figurename)
    print("los gehts")
    x,y = generateUmapForPlotting(embeddings_st1)
    print("test0")
    # By forum
    sns.set_theme(font_scale=font_scale, style = "white")
    plt.figure(figsize = (12.5,6))
    plt.xlim(-5,15)
    plt.ylim(-2, 10)
    plt.scatter(x, y, s=1, c = dataframe["forum_nr"].map(graph_colors_dict), hatch = "/")
    plt.title('2D Projection of Sentence Embeddings')
    print("test1")
    pop_list = []
    for forum in forums_to_process:
        pop = mpatches.Patch(color=graph_colors_dict.get(forum), label=forum_names_dict.get(forum))
        pop_list.append(pop)
    plt.legend(handles=pop_list, loc="upper right")
    plt.savefig(f"{path_figures}forums{figurename}.pdf")
    
    
def plot_all_forums_by_language():
    sns.set_theme(font_scale=font_scale, style = "white")
    plt.figure(figsize = (15,15))
    plt.xlim(-5,15)
    plt.ylim(-10, 15)
    plt.scatter(x, y, s=3, c = dataframe["language"].map(graph_colors_dict))
    plt.title(f'2D Projection of Sentence Embeddings')
    pop_en = mpatches.Patch(color=graph_colors_dict.get("english"), label="English")
    pop_ru = mpatches.Patch(color=graph_colors_dict.get("russian"), label='Russian')
    plt.legend(handles=[pop_en,pop_ru], loc="upper right")    
    plt.savefig(f"{path_figures}forums{figurename}_color_by_language.pdf")

In [ ]:
def plot_df_as_subset(subset_size =500):
    subset_to_plot = pd.DataFrame()
    for forum_id in list(dataframe_all_forums.groupby(['forum_nr']).count().index):
        df = dataframe_all_forums[dataframe_all_forums['forum_nr']== forum_id].sample(n=subset_size)
        subset_to_plot = pd.concat([subset_to_plot, df], ignore_index=True, axis=0)

    #create embeddings
    embeddings_subset = splitEmbed(model_=model_st1, model_type_='sentence transformer', all_sentences_=subset_to_plot['body'], n_partitions_ = 1)
    
    #plot the subset
    plot_all_forums(dataframe = subset_to_plot, embeddings_st1 = embeddings_subset)

In [ ]:
plot_df_as_subset(500)

In [ ]:
plot_all_forums(dataframe_all_forums, embeddings_st1) #To plot only some forums, select these forums in forums_to_process

# Generate the embeddings and clustering for whole frame

In [ ]:
dataframe_all_forums['cluster'], umap_embeddings = generateUmapAndCluster(embeddings_st1, n_neighbors = 6,  n_components= 9, random_state = 1, min_cluster_size = 24)
dataframe_all_forums = vectorToColumn(df = dataframe_all_forums, vectors = umap_embeddings, column_name = 'all-mpnet-base-v2_reduced')
dataframe_all_forums['cluster_label'], clusters_dict_all_forums = text_cluster_to_label(data_=dataframe_all_forums)

clusters_dict = {}
for forum_id in subforums_to_recalc:
    df_dict[forum_id]['cluster'], umap_embeddings = generateUmapAndCluster(embeddings_dict[forum_id], n_neighbors = 6,  n_components= 9, random_state = 1, min_cluster_size = 24)
    df_dict[forum_id] = vectorToColumn(df = df_dict[forum_id], vectors = umap_embeddings, column_name = 'all-mpnet-base-v2_reduced')
    df_dict[forum_id]['cluster_label'], clusters_dict[forum_id] = text_cluster_to_label(data_=df_dict[forum_id])

### SAVE DATAFRAMES

In [ ]:
dataframe_all_forums.to_pickle(f'{path_outputs}dataframe_all_forums.pkl')

for forum_id in subforums_to_recalc:
    df_dict[forum_id].to_pickle(f'{path_outputs}dataframe_{str(forum_id)}.pkl')

# Excluding one language

In [ ]:
# RUSSIAN TEXT ONLY

dataframe_all_forums, df_dict = load_dfs()
dataframe_all_forums = dataframe_all_forums[dataframe_all_forums["language"] == "russian"]

#cleaning
dataframe_all_forums, dropped_rows = cleanRowsWithPercentageOfPunctuations(dataframe_all_forums, threshold = 12)
dataframe_all_forums['body']=dataframe_all_forums['body'].map(clean_text_local) # clean text
dataframe_all_forums = dataframe_all_forums[dataframe_all_forums["body"] != ""] # delete empty rows
dataframe_all_forums = cleanRowsWithPercentageOfNumbers(dataframe_all_forums, threshold = 100)  #clean strings with threshold% numbers

#embeddings
model_st1 = SentenceTransformer('all-mpnet-base-v2') #sentence-BERT model
embeddings_st1 = splitEmbed(model_=model_st1, model_type_='sentence transformer', all_sentences_=dataframe_all_forums['body'], n_partitions_ = round(len(list(dataframe_all_forums["body"]))/8000)) # devide the set so that one partition has about 5000-10000 partitions
dataframe_all_forums=vectorToColumn(df=dataframe_all_forums,vectors=embeddings_st1,column_name='all_mpnet_base_v2')

#clustering
dataframe_all_forums['cluster'], umap_embeddings = generateUmapAndCluster(embeddings_st1, n_neighbors = 6,  n_components= 9, random_state = 1, min_cluster_size = 24)
dataframe_all_forums = vectorToColumn(df = dataframe_all_forums, vectors = umap_embeddings, column_name = 'all-mpnet-base-v2_reduced')
dataframe_all_forums['cluster_label'], clusters_dict_all_forums = text_cluster_to_label(data_=dataframe_all_forums)

#saving
dataframe_all_forums.to_pickle(f'{path_outputs}dataframe_all_forums_russian_only_clustering.pkl')

In [ ]:
# ENGLISH TEXT ONLY

dataframe_all_forums, df_dict = load_dfs()
dataframe_all_forums = dataframe_all_forums[dataframe_all_forums["language"] == "english"]

#cleaning
dataframe_all_forums, dropped_rows = cleanRowsWithPercentageOfPunctuations(dataframe_all_forums, threshold = 12)
dataframe_all_forums['body']=dataframe_all_forums['body'].map(clean_text_local) # clean text
dataframe_all_forums = dataframe_all_forums[dataframe_all_forums["body"] != ""] # delete empty rows
dataframe_all_forums = cleanRowsWithPercentageOfNumbers(dataframe_all_forums, threshold = 100)  #clean strings with threshold% numbers

#embeddings
model_st1 = SentenceTransformer('all-mpnet-base-v2') #sentence-BERT model
embeddings_st1 = splitEmbed(model_=model_st1, model_type_='sentence transformer', all_sentences_=dataframe_all_forums['body'], n_partitions_ = round(len(list(dataframe_all_forums["body"]))/8000)) # devide the set so that one partition has about 5000-10000 partitions
dataframe_all_forums=vectorToColumn(df=dataframe_all_forums,vectors=embeddings_st1,column_name='all_mpnet_base_v2')

#clustering
dataframe_all_forums['cluster'], umap_embeddings = generateUmapAndCluster(embeddings_st1, n_neighbors = 6,  n_components= 9, random_state = 1, min_cluster_size = 24)
dataframe_all_forums = vectorToColumn(df = dataframe_all_forums, vectors = umap_embeddings, column_name = 'all-mpnet-base-v2_reduced')
dataframe_all_forums['cluster_label'], clusters_dict_all_forums = text_cluster_to_label(data_=dataframe_all_forums)

#saving
dataframe_all_forums.to_pickle(f'{path_outputs}dataframe_all_forums_english_only_clustering.pkl')